# Soul Home: scan → floor plan → Vastu zones (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prabhat-twr/Home-planner-/blob/main/notebooks/soulhome_colab.ipynb)

This notebook runs the full reconstruction pipeline from the [Home-planner-](https://github.com/prabhat-twr/Home-planner-) repo:

| Tier | Input | Models |
|---|---|---|
| Panorama | 360° photo sphere (2:1 JPG/PNG) | HorizonNet layout · Depth Anything V2 · YOLO11 |
| LiDAR | GLB / OBJ / PLY / USDZ mesh | geometry only (no GPU needed) |
| ARCore | PLY point cloud | geometry only |

**Before you start:** *Runtime → Change runtime type → T4 GPU.* The panorama models also run on CPU, about 10× slower.

## 1 · Setup

In [ ]:
import os, subprocess, sys
if not os.path.exists("Home-planner-"):
    !git clone -q https://github.com/prabhat-twr/Home-planner-
%cd Home-planner-
!{sys.executable} -m pip install -q -e ".[ml,usd,dev]"

In [ ]:
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU mode)")

Optional: run the regression suite (38 tests, ~15 s). Every capture tier must recover a known room to the centimetre.

In [ ]:
!{sys.executable} -m pytest -q

## 2 · Helpers
`show_result` prints the summary, draws the floor plan and lists objects with their Vastu direction and element.

In [ ]:
import json, shutil, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from IPython.display import SVG, display, Markdown

from soulhome.pipeline.run import run_scan, ScanMeta
from soulhome.spatial.zones import zone_report

WORK = Path("/content/scans") if Path("/content").exists() else Path("scans")
WORK.mkdir(parents=True, exist_ok=True)

def show_result(model, out_dir):
    room = model.rooms[0]
    display(Markdown(
        f"### {room.name}\n"
        f"**{room.floor_area_m2:.2f} m²** floor · **{room.ceiling_height_m:.2f} m** ceiling · "
        f"{len(room.walls)} walls · {sum(o.type=='door' for o in room.openings)} doors · "
        f"{sum(o.type=='window' for o in room.openings)} windows · {len(room.objects)} objects · "
        f"confidence **{model.quality.confidence:.0%}**"))
    for w in model.quality.warnings:
        print("⚠", w)
    display(SVG(filename=str(Path(out_dir) / "floorplan.svg")))
    if room.objects:
        display(pd.DataFrame([{
            "object": o.label, "element": o.element, "direction": o.direction_16, "grid zone": o.grid_zone,
            "x (m)": o.position[0], "y (m)": o.position[1], "confidence": o.confidence, "method": o.placement_method,
        } for o in room.objects]))
    display(pd.DataFrame([{"stage": s.name, "status": s.status, "ms": s.duration_ms} for s in model.provenance.stages]))

def fresh(name):
    d = WORK / name
    shutil.rmtree(d, ignore_errors=True)
    (d / "in").mkdir(parents=True)
    return d

## 3 · Synthetic demos (no models)
An analytic L-shaped room (21.25 m², 2.70 m ceiling, one door, one window). The LiDAR mesh is noisy, rotated 17°, Y-up and has floating fragments; the pipeline still recovers it exactly.

In [ ]:
from soulhome.demo import write_mesh_demo, write_panorama_demo

for tier in ["lidar", "arcore"]:
    d = fresh(f"demo_{tier}")
    meta = write_mesh_demo(d / "in", tier)
    model = run_scan(f"demo_{tier}", tier, d / "in", d / "out", ScanMeta(**meta))
    show_result(model, d / "out")

## 4 · Load the GPU models
HorizonNet (official MP3D weights from the HF Hub, loaded with the safe `weights_only` loader), Depth Anything V2 metric-indoor, YOLO11s. ~250 MB on first run.

In [ ]:
import time
from soulhome.inference.models import TorchInference
t = time.time()
inference = TorchInference()
print(f"loaded in {time.time()-t:.1f}s on {inference.device}:", inference.models)

## 5 · Real panorama: the HorizonNet sample bedroom

In [ ]:
SAMPLE = "https://raw.githubusercontent.com/sunset1995/HorizonNet/master/assets/demo.png"
d = fresh("sample_bedroom")
urllib.request.urlretrieve(SAMPLE, d / "in" / "panorama.png")
rgb = cv2.cvtColor(cv2.imread(str(d / "in" / "panorama.png")), cv2.COLOR_BGR2RGB)

def show_layout(rgb, layout):
    """Overlay HorizonNet's wall-ceiling (green) and wall-floor (red) boundaries."""
    ceil_lat, floor_lat, corner = layout
    h, w = rgb.shape[:2]
    cols = np.linspace(0, w - 1, len(floor_lat))
    plt.figure(figsize=(16, 8))
    plt.imshow(rgb)
    for lat, c in ((ceil_lat, "lime"), (floor_lat, "red")):
        plt.plot(cols, (np.pi / 2 - lat) / np.pi * h, c, lw=2)
    for u in np.nonzero(corner > 0.5)[0]:
        plt.axvline(cols[u], color="cyan", lw=0.5, alpha=0.5)
    plt.axis("off"); plt.show()

show_layout(rgb, inference.layout(cv2.resize(rgb, (1024, 512))))
model = run_scan("sample_bedroom", "panorama", d / "in", d / "out",
                 ScanMeta(camera_height_m=1.6, north_deg=0, room_name="bedroom (sample)"), inference=inference)
show_result(model, d / "out")

## 6 · Your own room
**Panorama:** use a **360° photo sphere**: a 2:1 image from a 360 camera, Pixel Photo Sphere or a 360 capture app. Stand near the middle of the room and set `CAMERA_HEIGHT` to how high you held the phone.

Phone **Pano-mode** strips (e.g. 10840×3784) are rejected by default. They miss the floor and ceiling edges the layout model measures from, so the room size would be wrong. Set `ALLOW_PARTIAL = True` only for a rough look.

`NORTH_DEG` is the compass bearing your phone faced at the *start/centre* of the capture (it drives the Vastu directions).

In [ ]:
CAMERA_HEIGHT = 1.5      # metres
NORTH_DEG = 0            # compass bearing of the panorama centre
ROOM_NAME = "my room"
ALLOW_PARTIAL = False    # True = process phone Pano strips anyway (low confidence)
PANO_HFOV = None         # phone Pano only: how far you turned, in degrees (e.g. 270 or 360)

d = fresh("my_panorama")
try:
    from google.colab import files
    uploaded = files.upload()                 # pick your panorama image
    name = next(iter(uploaded))
    (d / "in" / name).write_bytes(uploaded[name])
except ImportError:                           # not in Colab: fall back to the sample
    urllib.request.urlretrieve(SAMPLE, d / "in" / "panorama.png")

meta = ScanMeta(camera_height_m=CAMERA_HEIGHT, north_deg=NORTH_DEG, room_name=ROOM_NAME,
                allow_partial_panorama=ALLOW_PARTIAL, pano_hfov_deg=PANO_HFOV)
try:
    model = run_scan("my_panorama", "panorama", d / "in", d / "out", meta, inference=inference)
    show_result(model, d / "out")
except Exception as e:
    print("Scan failed:", e)

### LiDAR / ARCore upload
Export the scan from a 3D-scanner app (iPhone Pro LiDAR) as **GLB, OBJ, PLY or USDZ**. glTF/GLB is Y-up by spec; for OBJ/PLY exported Y-up set `UP_AXIS = "y"`. Units are auto-detected (m/cm/mm/in).

In [ ]:
TIER = "lidar"           # or "arcore"
UP_AXIS = None           # None = auto (GLB/glTF are Y-up), or "y" / "z"
NORTH_DEG = 0

d = fresh("my_mesh")
try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        (d / "in" / name).write_bytes(data)
except ImportError:
    write_mesh_demo(d / "in", TIER)

try:
    model = run_scan("my_mesh", TIER, d / "in", d / "out",
                     ScanMeta(north_deg=NORTH_DEG, up_axis=UP_AXIS, room_name="my room (LiDAR)"))
    show_result(model, d / "out")
except Exception as e:
    print("Scan failed:", e)

## 7 · Vastu zone queries
Objects and openings grouped by 16-direction sector (from the room centroid) or by the 3×3 Vastu grid (centre = Brahmasthan).

In [ ]:
for system in ["16", "grid"]:
    print(f"--- {system} ---")
    for room_id, zones in zone_report(model, system).items():
        for zone, items in sorted(zones.items()):
            print(f"{zone:12s}", ", ".join(f"{i['label'] if i['kind']=='object' else i['kind']}" for i in items))

## 8 · Outputs
`model.json` is the versioned spatial contract the rule engine consumes; `floorplan.svg` is the drawing; LiDAR scans also produce `mesh_clean.glb`.

In [ ]:
out = d / "out"
print(sorted(p.name for p in out.iterdir()))
print(json.dumps(json.loads((out / "model.json").read_text())["rooms"][0]["walls"][:2], indent=2))
try:
    from google.colab import files
    shutil.make_archive("/content/soulhome_result", "zip", out)
    files.download("/content/soulhome_result.zip")
except ImportError:
    pass

## 9 · (Optional) Use the deployed API instead
The same pipeline runs on Modal behind a REST API. A web UI is at https://soulhome-planner.netlify.app.

In [ ]:
import requests, time
API = "https://mptmt16--soulhome-api.us-east.modal.run"
job = requests.post(f"{API}/v1/demo/lidar", timeout=60).json()
while job["status"] not in ("succeeded", "failed"):
    time.sleep(job.get("poll_after_s", 2))
    job = requests.get(f"{API}/v1/scans/{job['id']}", timeout=30).json()
print(job["status"], {k: v for k, v in (job.get("summary") or {}).items() if k != "stages"})
if job["status"] == "succeeded":
    display(SVG(requests.get(job["links"]["floorplan"], timeout=30).text))